# SA-Jev · 环境与数据准备
先跑 pilot。正式训练数据约 8,000 HotpotQA + 4,000 MuSiQue；每题最多 3 个状态。所有模型输入排除答案、分解和 gold 标签。

In [1]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', 'https://github.com/BoldHu/SA-Jev.git')  # Override with SAJEV_REPO_URL if needed.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'pilot'  # Change to 'full' only after the pilot completes and memory is measured.
# MODE = 'full'  
ROOT = DRIVE / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
print('Code:', PROJECT, 'Artifacts:', ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Code: /content/graphrag-jev Artifacts: /content/drive/MyDrive/SA-Jev/pilot


In [2]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# Activate editable-install .pth files in this already-running notebook kernel.
import importlib, site
site_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    site_dirs.append(site.getusersitepackages())
for site_dir in dict.fromkeys(site_dirs):
    site.addsitedir(site_dir)
importlib.invalidate_caches()
import sa_jev
print('SA-Jev source:', sa_jev.__file__)

import importlib, site

site_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    site_dirs.append(site.getusersitepackages())

for site_dir in dict.fromkeys(site_dirs):
    site.addsitedir(site_dir)

importlib.invalidate_caches()

import sa_jev
print('SA-Jev source:', sa_jev.__file__)

# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config()
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    from collections import deque
    tail = deque(maxlen=40)
    cmd = [sys.executable, '-u', '-m', 'sa_jev.cli', command, '--root', str(ROOT), *map(str,args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise RuntimeError(f'sajev {command} failed (exit {returncode}). Last output: ' + ''.join(tail))


SA-Jev source: /content/graphrag-jev/src/sa_jev/__init__.py
SA-Jev source: /content/graphrag-jev/src/sa_jev/__init__.py


In [3]:
# Read the same credential file in every notebook. Fill it in once.
from sa_jev.environment import load_environment
ENV_FILE = (DRIVE if 'google.colab' in sys.modules else PROJECT) / '.env'
credential_status = load_environment(ENV_FILE, template=PROJECT / '.env.example')
print('Credential file:', ENV_FILE)
for name in ('OPENAI_API_KEY', 'OPENAI_BASE_URL', 'TYPESAFE_API_KEY', 'HF_TOKEN'):
    print(name + ':', 'loaded' if credential_status[name] else 'empty')
if not credential_status['OPENAI_API_KEY']:
    print('Fill OPENAI_API_KEY in the file above, save it, then rerun this cell before building the index or answering questions.')


Credential file: /content/drive/MyDrive/SA-Jev/.env
OPENAI_API_KEY: loaded
OPENAI_BASE_URL: loaded
TYPESAFE_API_KEY: loaded
HF_TOKEN: loaded


In [4]:
import torch
from pathlib import Path
from tempfile import mkdtemp
from sa_jev.smoke import run

smoke_parent = ROOT / "smoke"
smoke_parent.mkdir(parents=True, exist_ok=True)

smoke_dir = Path(mkdtemp(prefix="run_", dir=smoke_parent))
print("本次测试目录：", smoke_dir)

summary = run(
    smoke_dir,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
summary

本次测试目录： /content/drive/MyDrive/SA-Jev/pilot/smoke/run_t7qzp2c3
{'step': 1, 'epoch': 0, 'rank': 1.7917581796646118, 'suf': 0.6931471228599548, 'rel': 0.6931471526622772, 'dpo': 0.0, 'stability': 0.0, 'allocated_gib': 0.017200946807861328, 'reserved_gib': 0.041015625, 'peak_allocated_gib': 0.025798320770263672, 'peak_reserved_gib': 0.041015625, 'seconds': 0.8819468530000449, 'eta_seconds': 0.8819468530000449, 'grad_norm': 0.00011680091120069847, 'valid_rank': 1, 'valid_suf': 2, 'valid_rel': 10, 'task': 2.623534733057022, 'total_loss': 2.623534733057022}
{'step': 2, 'epoch': 0, 'rank': 1.6094295978546143, 'suf': 0.6931471824645996, 'rel': 0.693133682012558, 'dpo': 0.0, 'stability': 0.0, 'allocated_gib': 0.017200946807861328, 'reserved_gib': 0.04296875, 'peak_allocated_gib': 0.02643585205078125, 'peak_reserved_gib': 0.04296875, 'seconds': 1.0194010160003018, 'eta_seconds': 0.0, 'grad_norm': 0.00025760586140677333, 'valid_rank': 1, 'valid_suf': 2, 'valid_rel': 9, 'task': 2.4412035167217256,

{'test_only': True,
 'output_directory': '/content/drive/MyDrive/SA-Jev/pilot/smoke/run_t7qzp2c3',
 'model': 'tiny causal encoder + actual upstream OpenJev DecisionHead/OptionSetInteractor',
 'data': 'synthetic fixtures',
 'device': 'cuda',
 'train_states': 6,
 'validation_states': 3,
 'preference_pairs': 6,
 'stage1_optimizer_steps': 2,
 'stage2_optimizer_steps': 2,
 'checkpoint_roundtrip': True,
 'evaluation_rows': 1,
 'evaluation_errors': 0,
 'peak_reserved_gib': 0.04296875,
 'not_verified': ['trained OpenJev-4B weights',
  'A100 80GB peak',
  'live LightRAG extraction',
  'official Jev API',
  'scientific performance']}

下载原始数据：pilot 每个文件 64 条；full 读取完整源。2Wiki 的官方 dev.json 另置于 ROOT/raw/2wiki_dev.json。MuSiQue 两种官方格式均受支持。

In [5]:
from sa_jev.preparation import download_sources, download_2wiki, normalize_sources
download_sources(ROOT/'raw', limit=64 if MODE=='pilot' else None)
if MODE=='full':
    download_2wiki(ROOT/'raw')
questions, manifest = normalize_sources(ROOT/'raw', ROOT/'data')
manifest

{'sources': [{'path': 'hotpot_train.jsonl',
   'records': 64,
   'sha256_content': 'dec5aaa205225ab5efc20297301c992ada64badd3514f05a5198da3b41abd763'},
  {'path': 'hotpot_validation.jsonl',
   'records': 64,
   'sha256_content': '7e401a283acaf799a1acbfe1e0449991d00e51237cd31787a45714ec16001d2f'},
  {'path': 'musique_ans_train.jsonl',
   'records': 64,
   'sha256_content': '584e8c30fe87d6aebe63fbf8696e5444dc22695be21d9f6cb2cd0619f12982b7'},
  {'path': 'musique_ans_dev.jsonl',
   'records': 64,
   'sha256_content': 'aaab4c1a602d9bebae91221737c6bc13d137ba5c0200bbbf6cbcdc2426941d07'}],
 'question_fingerprint': '6443f076514e3967c9e1d0ebae736a295ace806bb0faa7cdccac195039e86c57',
 'corpus_fingerprint': '9fe88ac93de0f308187ecc510fd773c9858fcecaacef3bd0a2644238b77b8018',
 'splits': {'train': 111, 'validation': 12, 'test': 128, 'heldout': 5},
 'seed': 17,
 'family_links': 0,
 'corpus_protocol': 'union of raw source paragraphs; no answers, decompositions or support labels indexed',
 'scope': 'bou

固定 LightRAG 混合召回。原始段落统一入库，不索引答案或标签。此步骤会调用所配置的抽取与嵌入服务；先检查 configs/research.yaml。首次建图通常比训练 pilot 更耗时。

In [ ]:
from sa_jev.experiments import cache_pools
await cache_pools(ROOT, config)

INFO: Creating working directory /content/drive/MyDrive/SA-Jev/pilot/index
INFO: [] Created new empty graph file: /content/drive/MyDrive/SA-Jev/pilot/index/graph_chunk_entity_relation.graphml
INFO: Role LLM Configuration (initialized):
INFO:  - extract: None/None, host=None, max_async=4, timeout=240
INFO:  - keyword: None/None, host=None, max_async=4, timeout=240
INFO:  - query: None/None, host=None, max_async=4, timeout=240
INFO:  - vlm: None/None, host=None, max_async=4, timeout=240
INFO: [] Process 18857 KV load full_docs with 0 records
INFO: [] Process 18857 KV load text_chunks with 0 records
INFO: [] Process 18857 KV load full_entities with 0 records
INFO: [] Process 18857 KV load full_relations with 0 records
INFO: [] Process 18857 KV load entity_chunks with 0 records
INFO: [] Process 18857 KV load relation_chunks with 0 records
INFO: [] Process 18857 KV load llm_response_cache with 0 records
INFO: [] Process 18857 doc status load doc_status with 0 records
INFO: Processing 32 doc

indexed 32/3450 documents


INFO: Processing 32 document(s)
INFO: Parsing (native): e_54d9c44b9282048a9c75
INFO: Parsing (native): e_c9e434e04f3c60b26964
INFO: Parsing (native): e_4c81f5d16f3b4f98c1de
INFO: Parsing (native): e_a68b2fd93ab99a6ba7f4
INFO: Parsing (native): e_dcd4d4a2b813a3fca287
INFO: Parsing (native): e_74c7c514d69813c6008b
INFO: Parsing (native): e_6bc9233f125caccdfa3a
INFO: Parsing (native): e_91352af018c3225f9cc8
INFO: Parsing (native): e_02c17f0217bdc5d10c91
INFO: Parsing (native): e_3cb26fea8bb6c6acfb0b
INFO: Extracting stage 1/32: e_54d9c44b9282048a9c75
INFO: Processing d-id: e_54d9c44b9282048a9c75
INFO: Extracting stage 2/32: e_c9e434e04f3c60b26964
INFO: Processing d-id: e_c9e434e04f3c60b26964
INFO: Chunking F(legacy): size=512, split_only=False, overlap=64, doc_id: e_54d9c44b9282048a9c75
INFO: Chunking F(legacy): size=512, split_only=False, overlap=64, doc_id: e_c9e434e04f3c60b26964
INFO: Extracting stage 3/32: e_4c81f5d16f3b4f98c1de
INFO: Processing d-id: e_4c81f5d16f3b4f98c1de
INFO: Chun

indexed 64/3450 documents


INFO: Processing 32 document(s)
INFO: Parsing (native): e_918230b79dda2e8decd1
INFO: Parsing (native): e_cf2303bfe7067df2ef8c
INFO: Parsing (native): e_cd141b54ef0e0bb082f4
INFO: Parsing (native): e_77f6dcaccd58ad35125f
INFO: Parsing (native): e_37dd12f57132b584dce3
INFO: Parsing (native): e_dbb24c63081235ec386f
INFO: Parsing (native): e_2b4e0bf7269008afdb79
INFO: Parsing (native): e_09b937bad4850aa54a59
INFO: Parsing (native): e_e6f33ff5a937182e776d
INFO: Parsing (native): e_a6e6fc7aa0aa89ddadc4
INFO: Extracting stage 1/32: e_918230b79dda2e8decd1
INFO: Processing d-id: e_918230b79dda2e8decd1
INFO: Extracting stage 2/32: e_cf2303bfe7067df2ef8c
INFO: Processing d-id: e_cf2303bfe7067df2ef8c
INFO: Chunking F(legacy): size=512, split_only=False, overlap=64, doc_id: e_918230b79dda2e8decd1
INFO: Chunking F(legacy): size=512, split_only=False, overlap=64, doc_id: e_cf2303bfe7067df2ef8c
INFO: Extracting stage 3/32: e_cd141b54ef0e0bb082f4
INFO: Processing d-id: e_cd141b54ef0e0bb082f4
INFO: Chun

indexed 96/3450 documents


... (14 outputs left out)

INFO: Processing 32 document(s)
INFO: Parsing (native): e_a3773ff54d1dad064368
INFO: Parsing (native): e_020e0c2bde4d52cba02e
INFO: Parsing (native): e_16102f55684be951f0a7
INFO: Parsing (native): e_f11024b703a30fe45900
INFO: Parsing (native): e_9f9a028072781970d0c2
INFO: Parsing (native): e_ba53347ca645ba55a18c
INFO: Parsing (native): e_2beda20da35d0f373805
INFO: Parsing (native): e_12508580991f93fbba25
INFO: Parsing (native): e_668e451ef3e0d2e2df7d
INFO: Parsing (native): e_f5092b3789e48ac221e4
INFO: Extracting stage 1/32: e_a3773ff54d1dad064368
INFO: Processing d-id: e_a3773ff54d1dad064368
INFO: Extracting stage 2/32: e_020e0c2bde4d52cba02e
INFO: Processing d-id: e_020e0c2bde4d52cba02e
INFO: Chunking F(legacy): size=512, split_only=False, overlap=64, doc_id: e_a3773ff54d1dad064368
INFO: Chunking F(legacy): size=512, split_only=False, overlap=64, doc_id: e_020e0c2bde4d52cba02e
INFO: Extracting stage 3/32: e_16102f55684be951f0a7
INFO: Processing d-id: e_16102f55684be951f0a7
INFO: Chun

CancelledError: 

ERROR: Embedding func: Error in decorated function for task 136267669136576_4778.710723642: RetryError[<Future at 0x7bef4a1c79d0 state=finished raised RateLimitError>]


ERROR: Embedding func: Error in decorated function for task 136267669142208_4778.711072501: RetryError[<Future at 0x7bef4a1e26d0 state=finished raised RateLimitError>]
ERROR: Embedding func: Error in decorated function for task 136267669131472_4778.710895752: RetryError[<Future at 0x7bef4a1e03d0 state=finished raised RateLimitError>]


In [ ]:
from sa_jev.preparation import prepare_tasks
prepare_tasks(ROOT/'data', counts=config['data']['train_counts'], mine=True, dense=config['data']['dense_mining'])

训练前完整扫描公开 OpenJevData。检出的重叠保留记录并在论文报告；未检出不代表证明完全未见。此步骤不使用独立扰动测试集调参。

In [ ]:
cli('overlap')
cli('review-packets')

In [ ]:
from sa_jev.io import read_json
read_json(ROOT/'data/task_manifest.json')

核对实际 K 分布和有效标签数量。缺候选不复制补齐，未知相关性不硬判负例。若排序有效状态太少，核查候选注释映射/补充核验，不能把掩码简单删掉。把独立扰动草稿人工核验后保存为 perturbations_reviewed.jsonl。